In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Data Reading


In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@adbstoragepro.dfs.core.windows.net/customer")
display(df)


In [0]:
df = df.drop("_rescued_data")
display(df)

In [0]:
df = df.withColumn("domains", split(df["email"], "@")[1])
display(df)

In [0]:
display(
    df.groupBy("domains")
      .agg(count("customer_id").alias("total_customer"))
      .sort("total_customer", ascending=False)
)

In [0]:
df_gmail = df.filter(col('domains') == "gmail.com")
display(df_gmail)
time.sleep(6)

df_gmail = df.filter(col('domains') == "yahoo.com")
display(df_gmail)
time.sleep(5)

df_gmail = df.filter(col('domains') == "hotmail.com")
display(df_gmail)
time.sleep(5)

In [0]:
df = df.withColumn("full_name", concat(col("first_name"), lit(' '), col("last_name")))
display(df)

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@adbstoragepro.dfs.core.windows.net/customer")

In [0]:
%sql
CREATE SCHEMA DEV.SILVER;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS DEV.SILVER.customer_silver
    USING DELTA
    LOCATION 'abfss://silver@adbstoragepro.dfs.core.windows.net/customer';


In [0]:
%sql
SELECT * FROM DEV.SILVER.customer_silver;